### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="regensburg_pediatric_appendicitis",
    dataset_year="2021",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://doi.org/10.5281/zenodo.7711412",
    download_description="""
We get the newest version of the tabular data from Zenodo.


wget https://zenodo.org/records/7711412/files/app_data.xlsx?download=1
mkdir -p local-data-warehouse/regensburg_pediatric_appendicitis && mv app_data.xlsx?download=1 local-data-warehouse/regensburg_pediatric_appendicitis/
""",
    # References
    academic_reference_bibtex=r"""@article{marcinkevivcs2024interpretable,
  title={Interpretable and intervenable ultrasonography-based machine learning models for pediatric appendicitis},
  author={Marcinkevi{\v{c}}s, Ri{\v{c}}ards and Wolfertstetter, Patricia Reis and Klimiene, Ugne and Chin-Cheong, Kieran and Paschke, Alyssia and Zerres, Julia and Denzinger, Markus and Niederberger, David and Wellmann, Sven and Ozkan, Ece and others},
  journal={Medical image analysis},
  volume={91},
  pages={103042},
  year={2024},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="marcinkevivcs2024interpretable",
    license="CC-BY-NC-4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the tabular meta-data used in the paper. We create only one task out of the dataset using one of the three available classes. We note that the main use case of the paper was image-based modelling and comparing to tabular baselines with image-to-tabular-radiometric. The authors also published the tabular data, which we use here.

- Note, for this dataset is was hard to choose the label given the tabular data. We picked "Severity" in the end. "Diagnosis" is a noisy ground truth for all cases but the ones that had surgery. It was either deterministically determined by the AS score, or during surgery. "Management" reflects the decision and opinion of senior pediatric surgeon. Only "Severity" represents a ground truth as all cases that were server, were either clearly servery or did not even have appendicitis.
 - Warning: We tried to clean the data as good as possible but we suspect it might still contain some features leaking the severity, but we lack the medical expertise to identify them. Thus, I would not be surprised if we had to remove this dataset later again.
- The data contains string data, but only for a very small number of patients and the text is often seemingly very useless. The features add most likely no value, but we keep them in for the pipeline/model to ignore if needed.
- We drop the few patients that did not have an ultrasound to avoid them being leaking due to missing features.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Severity",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Severity",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel(dataset_mold.path / "app_data.xlsx?download=1", sheet_name=0)
print("Loaded data shape:", df.shape)

# Drop samples w/o utrasounds
df = df[df["US_Performed"] == "yes"]

as_cat_type = [
    "Sex",
    "Severity",
    "Peritonitis",
    "Migratory_Pain",
    "Lower_Right_Abd_Pain",
    "Contralateral_Rebound_Tenderness",
    "Ipsilateral_Rebound_Tenderness",
    "Coughing_Pain",
    "Psoas_Sign",
    "Nausea",
    "Loss_of_Appetite",
    "Dysuria",
    "Stool",
    "Ketones_in_Urine",
    "RBC_in_Urine",
    "WBC_in_Urine",
    "Neutrophilia",
    "Appendix_on_US",
    "Free_Fluids",
    "Appendix_Wall_Layers",
    "Target_Sign",
    "Perfusion",
    "Surrounding_Tissue_Reaction",
    "Pathological_Lymph_Nodes",
    "Bowel_Wall_Thickening",
    "Ileus",
    "Coprostasis",
    "Meteorism",
    "Enteritis",
    "Appendicolith",
    "Perforation",
    "Appendicular_Abscess",
    "Conglomerate_of_Bowel_Loops",
]
as_string_type = [
    "Lymph_Nodes_Location",
    "Abscess_Location",
    "Gynecological_Findings"
]
df = df.drop(columns=[
    # Other
    "US_Performed", # constant after dropping other cases
    "US_Number", # rel us_performed
    "Length_of_Stay", # only available at discharge (so after diagnosis)
    # Other target labels Diagnosis / Management / Severity
    "Diagnosis_Presumptive",
    "Diagnosis",
    "Management",
])

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()